In [1]:
import pandas as pd

In [2]:
df  = pd.read_csv('../data/comentarios_toxicos_ptBR.csv')
df.head()

,Unnamed: 0,text,text_norm,toxic
0,12451,brocolis é muito boooom pqp,brócolis boom puta pariu,0
1,4080,to correndo disso,to correndo disso,0
2,5195,presidente da oab sofre amea莽as de morte e aci...,presidente aba sofre ame-a morte aciona polega...,0
3,13375,@user @user aí desanima pq para de vir teus ch...,aí desanima porque vir champ chato pra caralho...,1
4,4130,"ser feio e beta é um bosta, puta que me pariu....",feio beta bosta puta pariu lá vou comprar ócul...,1


In [3]:
df = df.drop(columns=['text_norm'])
df.head()

,Unnamed: 0,text,toxic
0,12451,brocolis é muito boooom pqp,0
1,4080,to correndo disso,0
2,5195,presidente da oab sofre amea莽as de morte e aci...,0
3,13375,@user @user aí desanima pq para de vir teus ch...,1
4,4130,"ser feio e beta é um bosta, puta que me pariu....",1


In [4]:
df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29921 entries, 0 to 29921
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  29921 non-null  int64 
 1   text        29921 non-null  object
 2   toxic       29921 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 935.0+ KB


In [5]:
df['toxic'].value_counts()

toxic
0    16412
1    13509
Name: count, dtype: int64

In [ ]:
import re
import spacy
from spacy.tokenizer import Tokenizer
from spacy.lang.pt import Portuguese
nlp = Portuguese()
tokenizer = Tokenizer(nlp.vocab)

In [8]:
# Remover Storpword usando spaCy
nlp = spacy.load("pt_core_news_sm")
doc = nlp("O rato roeu a roupa do rei de Roma")

def remover_stopwords(lista_de_tokens):
    return [token for token in lista_de_tokens if not token.is_stop]

print(remover_stopwords(doc))

[rato, roeu, roupa, rei, Roma]


In [9]:
def preProcessar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúâêôãõüç\s]', '', texto)
    token = tokenizer(texto)
    stopwords_removidos = remover_stopwords(token)
    return stopwords_removidos

df['text_normalized'] = df['text'].apply(preProcessar)
print(df['text_normalized'].head())

0                              [brocolis, boooom, pqp]
1                                       [to, correndo]
2    [presidente, oab, sofre, ameaas, morte, aciona...
3    [user, user, desanima, pq, vir, champ, chato, ...
4    [feio, beta, bosta, puta, pariu, vou, comprar,...
Name: text_normalized, dtype: object


In [10]:
print(df['text'][0])
print(df['text_normalized'][0])

brocolis é muito boooom pqp
[brocolis, boooom, pqp]


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

In [12]:
df['text_normalized_str'] = df['text_normalized'].apply(lambda x: ' '.join([t.text for t in x]))
X = df['text_normalized_str']
y = df['toxic']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [14]:
tfidf = TfidfVectorizer(max_features=10000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

In [24]:
# Regressão Logística
model = LogisticRegression(random_state=42, )
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

target_names = ['0', '1']
print(classification_report(y_test, y_pred, target_names=target_names))

              precision    recall  f1-score   support

           0       0.76      0.85      0.80      3283
           1       0.79      0.67      0.73      2702

    accuracy                           0.77      5985
   macro avg       0.77      0.76      0.77      5985
weighted avg       0.77      0.77      0.77      5985



In [25]:
def previsao(text):
    input_text_normalized = preProcessar(text)
    input_text_normalized_str = ' '.join([t.text for t in input_text_normalized])
    input_vec = tfidf.transform([input_text_normalized_str])
    prediction = model.predict(input_vec)
    return prediction[0]

In [31]:
input_text = "Você é um idiota!"
resultado = previsao(input_text)
print(f"Texto: {input_text}")
print(f"Previsão: {'Tóxico' if resultado == 1 else 'Não Tóxico'}")

Texto: Você é um idiota!
Previsão: Tóxico
